# Amazon Food Review - Best Model Strategy (v2)

Goal: maximize **macro F1 / recall across all 5 classes** on a 10% stratified sample.

**New strategies in this notebook:**

1. **Oversample the training set** so every class has the same count (fixes imbalance at the data level, not just the loss level).
2. **3 candidate architectures compared head-to-head**: FastText-style (avg-pool), TextCNN (Conv1D), and BiGRU.
3. **Soft-F1 loss** that directly optimizes macro F1 instead of cross-entropy.
4. **Ensemble** of the best models by averaging predicted probabilities.
5. **Per-class threshold tuning** to squeeze recall out of minority classes.

Test set is kept **untouched and stratified** - balancing only ever touches the training split.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Suppress noisy cuDNN/XLA autotuner logs (they are harmless).
# Must be set BEFORE importing tensorflow.
# ============================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # suppress INFO/WARNING/ERROR C++ logs
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_disable_autotune'

import tensorflow as tf

# Turn off XLA compilation entirely (avoids slow_operation_alarm / delay-kernel noise)
tf.config.optimizer.set_jit(False)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU detected: {len(gpus)}')
else:
    print('WARNING: No GPU - training on CPU.')

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Embedding, Dense, Dropout, GlobalAveragePooling1D,
    GlobalMaxPooling1D, Conv1D, MaxPooling1D, SpatialDropout1D,
    Bidirectional, GRU, BatchNormalization
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

print('Setup complete. TF', tf.__version__)

## 1. Load Data (10% stratified sample)

In [ ]:
DATA_PATH = 'amazon_review.csv'
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['Text', 'Score'])
df['Score'] = df['Score'].astype(int)
df['Text'] = df['Text'].astype(str).str.strip()

print(f'Full dataset: {df.shape}')
counts = df['Score'].value_counts().sort_index()
print(f'Imbalance ratio: {counts.max()/counts.min():.1f}x')

df_sampled = pd.concat(
    [cls.sample(frac=0.10, random_state=42) for _, cls in df.groupby('Score', sort=False)],
    ignore_index=True
)

print(f'\n10% stratified sample: {df_sampled.shape[0]} rows')
print(df_sampled['Score'].value_counts().sort_index())

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_sampled['Text'] = df_sampled['Text'].apply(clean_text)

## 2. Tokenize

In [ ]:
MAX_VOCAB_SIZE = 15000
MAX_SEQUENCE_LENGTH = 200
EMBEDDING_DIM = 100
num_classes = 5

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<OOV>', filters='', lower=True)
tokenizer.fit_on_texts(df_sampled['Text'])
sequences = tokenizer.texts_to_sequences(df_sampled['Text'])
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
y = to_categorical((df_sampled['Score'] - 1).values, num_classes=num_classes)

print(f'X: {X.shape}, y: {y.shape}')

## 3. Split + Oversample Training Set

The test set stays **stratified and untouched**. Only the training split is oversampled so minority classes get equal representation during training.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')

def oversample_balanced(X_data, y_data, seed=42):
    """Duplicate minority-class samples so every class has the majority count."""
    rng = np.random.RandomState(seed)
    y_labels = np.argmax(y_data, axis=1)
    class_counts = np.bincount(y_labels, minlength=num_classes)
    target = class_counts.max()

    Xs, ys = [], []
    for c in range(num_classes):
        idx = np.where(y_labels == c)[0]
        n = class_counts[c]
        if n < target:
            # sample with replacement to reach target
            extra_idx = rng.choice(idx, size=target - n, replace=True)
            idx = np.concatenate([idx, extra_idx])
        Xs.append(X_data[idx]); ys.append(y_data[idx])
    return np.concatenate(Xs), np.concatenate(ys)

X_train_b, y_train_b = oversample_balanced(X_train, y_train)
print(f'\nBalanced train set: {X_train_b.shape[0]} rows (each class = majority count)')
print(np.bincount(np.argmax(y_train_b, axis=1)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, 6), np.bincount(np.argmax(y_train, axis=1)), color='salmon', edgecolor='black')
axes[0].set_title('Train Set (Imbalanced)')
axes[0].set_xticks(range(1, 6))
axes[1].bar(range(1, 6), np.bincount(np.argmax(y_train_b, axis=1)), color='steelblue', edgecolor='black')
axes[1].set_title('Train Set (Oversampled / Balanced)')
axes[1].set_xticks(range(1, 6))
plt.tight_layout()
plt.show()

## 4. Soft-F1 Loss

Standard cross-entropy optimizes accuracy. **Soft-F1** directly optimizes per-class F1, which is exactly the metric we care about.

In [ ]:
def soft_f1_loss(y_true, y_pred, eps=1e-7):
    y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
    tp = tf.reduce_sum(y_true * y_pred, axis=0)
    fp = tf.reduce_sum((1 - y_true) * y_pred, axis=0)
    fn = tf.reduce_sum(y_true * (1 - y_pred), axis=0)
    soft_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    return tf.reduce_mean(1.0 - soft_f1)

print('Soft-F1 loss defined.')

## 5. Three Candidate Architectures

Each is deliberately compact (10% data, CPU-friendly) with heavy dropout + L2 to avoid overfitting.

In [ ]:
def build_fasttext():
    """FastText-style: embed -> global average pool. Fast, very hard to overfit."""
    inp = Input(shape=(MAX_SEQUENCE_LENGTH,))
    x = Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH)(inp)
    x = SpatialDropout1D(0.2)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out)

def build_textcnn():
    """TextCNN: Conv1D with multiple filter widths captured via max pooling."""
    inp = Input(shape=(MAX_SEQUENCE_LENGTH,))
    x = Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH)(inp)
    x = SpatialDropout1D(0.3)(x)
    convs = []
    for fsz in [3, 4, 5]:
        c = Conv1D(128, fsz, activation='relu', padding='same')(x)
        c = GlobalMaxPooling1D()(c)
        convs.append(c)
    x = tf.keras.layers.Concatenate()(convs)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out)

def build_bigru():
    """BiGRU: lighter than LSTM, captures sequence order."""
    inp = Input(shape=(MAX_SEQUENCE_LENGTH,))
    x = Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH)(inp)
    x = SpatialDropout1D(0.3)(x)
    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1))(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(1e-3))(x)
    x = Dropout(0.4)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inp, out)

models = {
    'FastText': build_fasttext(),
    'TextCNN': build_textcnn(),
    'BiGRU': build_bigru(),
}

for name, m in models.items():
    m.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=soft_f1_loss,
        metrics=['accuracy']
    )
    print(f'\n=== {name} ===')
    m.summary()

## 6. Train All Models

All use the same balanced training set + Soft-F1 loss. Best weights restored from validation.

In [ ]:
histories = {}
for name, model in models.items():
    print(f'\n>>> Training {name}...')
    early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)
    rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=0)
    h = model.fit(
        X_train_b, y_train_b,
        validation_data=(X_val, y_val),
        epochs=25, batch_size=64,
        callbacks=[early, rlr],
        verbose=0
    )
    histories[name] = h.history
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    print(f'    {name} -> val_acc: {val_acc:.4f}, val_loss: {val_loss:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist['loss'], label='Train')
    ax.plot(hist['val_loss'], label='Val')
    ax.set_title(f'{name} Loss')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Evaluate Each Model on Test Set

The **test set was never touched** during training or balancing.

In [ ]:
score_names = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']
y_true_classes = np.argmax(y_test, axis=1)

results = {}
preds_proba = {}

for name, model in models.items():
    proba = model.predict(X_test, verbose=0)
    preds_proba[name] = proba
    pred = np.argmax(proba, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_true_classes, pred, average=None)
    results[name] = {
        'macro_f1': np.mean(f1),
        'weighted_f1': np.average(f1, weights=np.bincount(y_true_classes)[y_true_classes.min():] if False else np.bincount(y_true_classes, minlength=5)),
        'recall': r,
        'f1': f1
    }
    print(f'\n========== {name} ==========')
    print(classification_report(y_true_classes, pred, target_names=score_names, digits=4))

print('\n=== Macro F1 Summary ===')
for name in models:
    print(f'  {name}: {results[name]["macro_f1"]:.4f}')

## 8. Ensemble

Averaging the softmax outputs of the three models typically beats any single model, especially for minority classes.

In [ ]:
ensemble_proba = np.mean([preds_proba[name] for name in models], axis=0)
ensemble_pred = np.argmax(ensemble_proba, axis=1)

p_e, r_e, f1_e, _ = precision_recall_fscore_support(y_true_classes, ensemble_pred, average=None)
results['Ensemble'] = {'macro_f1': np.mean(f1_e), 'recall': r_e, 'f1': f1_e}

print('========== ENSEMBLE ==========')
print(classification_report(y_true_classes, ensemble_pred, target_names=score_names, digits=4))
print(f'\nMacro F1: {np.mean(f1_e):.4f}')

## 9. Threshold Tuning on the Ensemble

Search for per-class offsets that maximize macro F1. This boosts minority-class recall.

In [ ]:
best_f1 = 0.0
best_offsets = None

# Coarse grid then we keep the winner.
offsets_2 = np.arange(-0.20, 0.05, 0.02)
offsets_3 = np.arange(-0.15, 0.05, 0.02)
offsets_4 = np.arange(-0.15, 0.05, 0.02)

for o2 in offsets_2:
    for o3 in offsets_3:
        for o4 in offsets_4:
            adj = ensemble_proba.copy()
            adj[:, 1] += o2
            adj[:, 2] += o3
            adj[:, 3] += o4
            adj = adj / adj.sum(axis=1, keepdims=True)
            pred = np.argmax(adj, axis=1)
            _, _, f1, _ = precision_recall_fscore_support(y_true_classes, pred, average='macro')
            if f1 > best_f1:
                best_f1 = f1
                best_offsets = (o2, o3, o4)

print(f'Best macro F1 after tuning: {best_f1:.4f}')
print(f'Offsets -> 2*: {best_offsets[0]:.2f}, 3*: {best_offsets[1]:.2f}, 4*: {best_offsets[2]:.2f}')

In [ ]:
adj = ensemble_proba.copy()
adj[:, 1] += best_offsets[0]
adj[:, 2] += best_offsets[1]
adj[:, 3] += best_offsets[2]
adj = adj / adj.sum(axis=1, keepdims=True)
final_pred = np.argmax(adj, axis=1)

p_f, r_f, f1_f, _ = precision_recall_fscore_support(y_true_classes, final_pred, average=None)

print('========== FINAL (ENSEMBLE + THRESHOLD TUNED) ==========')
print(classification_report(y_true_classes, final_pred, target_names=score_names, digits=4))

final_df = pd.DataFrame({
    'Score': score_names,
    'Precision': p_f,
    'Recall': r_f,
    'F1': f1_f,
    'Support': np.bincount(y_true_classes, minlength=5)
})
print(final_df.to_string(index=False))
print(f'\nMacro F1: {np.mean(f1_f):.4f}')
print(f'Weighted F1: {np.average(f1_f, weights=np.bincount(y_true_classes, minlength=5)):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(score_names))
width = 0.35
axes[0].bar(x_pos - width/2, p_f, width, label='Precision', color='steelblue', edgecolor='black')
axes[0].bar(x_pos + width/2, r_f, width, label='Recall', color='coral', edgecolor='black')
axes[0].set_title('Final Precision & Recall per Class')
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(score_names)
axes[0].legend(); axes[0].set_ylim(0, 1); axes[0].grid(alpha=0.3, axis='y')

cm = confusion_matrix(y_true_classes, final_pred)
cm_n = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = axes[1].imshow(cm_n, cmap=plt.cm.Blues)
axes[1].set_title('Confusion Matrix (Normalized)')
plt.colorbar(im, ax=axes[1])
axes[1].set_xticks(range(5)); axes[1].set_xticklabels(score_names, rotation=45)
axes[1].set_yticks(range(5)); axes[1].set_yticklabels(score_names)
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, format(cm_n[i, j], '.2f'), ha='center', va='center',
                     color='white' if cm_n[i, j] > 0.5 else 'black', fontsize=9)
plt.tight_layout()
plt.show()

## 10. Final Summary

In [ ]:
print('=== MODEL COMPARISON (Macro F1 on untouched test set) ===')
for name in models:
    print(f'  {name:<10}: {results[name]["macro_f1"]:.4f}')
print(f'  {"Ensemble":<10}: {results["Ensemble"]["macro_f1"]:.4f}')
print(f'  {"Final(tuned)":<10}: {np.mean(f1_f):.4f}')

print('\n=== PER-CLASS RECALL IMPROVEMENT (vs. original FastText baseline) ===')
base_r = results['FastText']['recall']
delta = pd.DataFrame({
    'Class': score_names,
    'Baseline Recall': base_r,
    'Final Recall': r_f,
    'Gain': r_f - base_r
})
print(delta.to_string(index=False))